# **Packaging and Project Structure**

**DS212 — Object-Oriented Programming**

**Week 13:** · **Suggested class time: 5 hours (2-hour lecture + 3-hour laboratory)**

**ILO :** Structure a distributable Python package with clear modules, metadata, dependencies, documentation, and version control.

In Week 8 we organized classes into **packages** inside one project. This week we go one step further: turning a package into something **other people can install** with `pip`, understand from its documentation, and trust because its history is tracked in version control.

```{mermaid}
flowchart LR
    A["Notebook code"] --> B["Modules & package<br/>(Week 8)"]
    B --> C["Distributable project<br/>pyproject.toml, README,<br/>LICENSE, tests"]
    C --> D["Version-controlled<br/>releases (Git tags)"]
    D --> E["Testing & distribution<br/>(Week 14)"]
```

Throughout this lesson and the next, we use a small example project, **`ds212-gradebook`**. Download the complete project as {download}`gradebook-project.zip <../files/gradebook-project.zip>` to follow along, or browse its structure below.

## **Package and Module Organization**

### **The `src` layout**

The Python Packaging User Guide recommends placing the package inside a `src/` folder:

```text
gradebook-project/               ← project root (the Git repository)
├── pyproject.toml               ← metadata, dependencies, build settings
├── README.md                    ← what it is and how to use it
├── LICENSE                      ← how others may use your code
├── CHANGELOG.md                 ← what changed in each version
├── .gitignore                   ← files Git should not track
├── src/
│   └── gradebook/               ← the importable package
│       ├── __init__.py          ← public interface + __version__
│       ├── exceptions.py        ← GradebookError and subclasses
│       ├── models.py            ← Student
│       └── gradebook.py         ← Gradebook
└── tests/                       ← tests live OUTSIDE the package
    ├── test_models.py
    └── test_gradebook.py
```

| Layout          | Package location     | Why it matters                                                                 |
| --------------- | -------------------- | ------------------------------------------------------------------------------ |
| **flat**        | `gradebook/` at root | simple, but tests may accidentally import the **local folder** instead of the installed package |
| **src**         | `src/gradebook/`     | forces you to **install** the package before using it, so tests check what users actually get |

The project root is **not** the package. It contains the package plus everything needed to build, test, document, and release it.

In [ ]:
from pathlib import Path

project = Path("..") / "files" / "gradebook-project"


def tree(folder, prefix=""):
    entries = sorted(p for p in folder.iterdir() if p.name != "__pycache__")
    for i, entry in enumerate(entries):
        last = i == len(entries) - 1
        print(prefix + ("└── " if last else "├── ") + entry.name)
        if entry.is_dir():
            tree(entry, prefix + ("    " if last else "│   "))


print(project.resolve().name)
tree(project)

### **Organizing modules inside the package**

Apply what you learned about responsibilities:

* one module per **cohesive group** of classes (`models.py`, `exceptions.py`);
* exceptions in their own module, so every other module can import them without circular imports;
* `__init__.py` re-exports the **public API** and defines `__version__`;
* internal helpers start with `_` (for example, `_parsing.py`) to signal "not part of the public API".

In [ ]:
import sys

sys.path.insert(0, str(project / "src"))     # in a real project, use: pip install -e .

import gradebook
from gradebook import Gradebook, Student

print("version:", gradebook.__version__)
print("public API:", gradebook.__all__)

book = Gradebook("DS212")
book.add(Student("2026-001", "Ana"))
book.record("2026-001", 92)
print(book.average("2026-001"), "2026-001" in book, len(book))

## **`pyproject.toml`: Metadata and Configuration**

`pyproject.toml` is the **single configuration file** for modern Python projects (PEP 621). It tells build tools how to build the package and tells `pip` and PyPI what the package is.

```toml
[build-system]                         # HOW to build
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.build_meta"

[project]                              # WHAT the package is
name = "ds212-gradebook-yourname"      # distribution name on PyPI (must be unique)
version = "0.1.0"
description = "A small object-oriented gradebook used in DS212."
readme = "README.md"
requires-python = ">=3.9"
license = { text = "MIT" }
authors = [{ name = "Your Name", email = "you@example.com" }]
dependencies = []                      # needed to RUN the package

[project.optional-dependencies]
dev = ["pytest>=7", "build", "twine"]  # needed only to DEVELOP it

[tool.setuptools.packages.find]        # tool-specific settings
where = ["src"]
```

Python can read TOML with the standard library, which is a handy way to check the file:

In [ ]:
import tomllib

with open(project / "pyproject.toml", "rb") as f:
    config = tomllib.load(f)

meta = config["project"]
print(meta["name"], meta["version"])
print("requires Python", meta["requires-python"])
print("dev extras:", meta["optional-dependencies"]["dev"])
print("build backend:", config["build-system"]["build-backend"])

### **Distribution name vs. import name**

| Name                 | Where it is used        | Example                     |
| -------------------- | ----------------------- | --------------------------- |
| **distribution name** | `pip install …`, PyPI  | `ds212-gradebook-yourname`  |
| **import name**      | `import …` in Python    | `gradebook`                 |

They often differ — `pip install scikit-learn` but `import sklearn`.

## **README, Requirements, and Dependency Management**

### **Dependencies**

| File / field                         | Purpose                                                   | Example                          |
| ------------------------------------ | --------------------------------------------------------- | -------------------------------- |
| `dependencies` in `pyproject.toml`   | what the **package** needs; use flexible ranges           | `"pandas>=2.0"`                  |
| `optional-dependencies`              | extras for development, docs, plotting                    | `pip install -e ".[dev]"`        |
| `requirements.txt`                   | exact versions for **reproducing an environment** (an app, a notebook analysis) | `pandas==2.2.3`     |

A library should not pin exact versions (it would conflict with other packages), while an **application or analysis** should pin them so results are reproducible.

```bash
python -m venv .venv
.venv\Scripts\activate          # Windows
pip install -e ".[dev]"         # editable install of your package + dev tools
pip freeze > requirements.txt   # snapshot of the exact environment
```

### **The README**

The README is the **front page** of your project — the first (and often only) thing people read. A good README answers:

1. **What** is this, and what problem does it solve?
2. **How** do I install it?
3. **How** do I use it? (a short, runnable example)
4. **How** do I run the tests or contribute?
5. **Who** made it, and under what **license**?

### **License**

Without a license, others legally **cannot** reuse your code. Common open-source choices are **MIT** (very permissive) and **Apache-2.0** (permissive, with patent terms). State the license in `LICENSE` and in `pyproject.toml`.

## **Metadata and Configuration Files**

| File              | Purpose                                                    |
| ----------------- | ---------------------------------------------------------- |
| `pyproject.toml`  | package metadata, dependencies, build and tool settings    |
| `README.md`       | documentation front page (also shown on PyPI)              |
| `LICENSE`         | legal terms for reuse                                      |
| `CHANGELOG.md`    | human-readable history of changes per version              |
| `.gitignore`      | files Git should ignore: `.venv/`, `__pycache__/`, `dist/` |
| `tests/`          | automated tests (Week 14)                                  |

## **Version Control and Release Preparation**

### **Semantic versioning**

Versions follow **MAJOR.MINOR.PATCH** (Semantic Versioning):

| Change                                         | Example          |
| ---------------------------------------------- | ---------------- |
| bug fix, nothing else changes                  | `0.1.0 → 0.1.1`  |
| new feature, existing code still works         | `0.1.1 → 0.2.0`  |
| **breaking** change to the public API          | `0.2.0 → 1.0.0`  |

Versions starting with `0.` signal that the API may still change.

### **A Git workflow for a package**

```bash
git init
git add .
git commit -m "Add Student and Gradebook classes"

git switch -c feature/export-csv        # work on a branch
# ... edit, test ...
git commit -am "Add CSV export"
git switch main
git merge feature/export-csv

git tag -a v0.2.0 -m "Release 0.2.0"    # mark a release
git push origin main --tags
```

```{mermaid}
gitGraph
    commit id: "Student, Gradebook"
    commit id: "tests"
    branch export-csv
    checkout export-csv
    commit id: "CSV export"
    checkout main
    merge export-csv tag: "v0.2.0"
```

### **Release checklist**

Before tagging a release:

* [ ] all tests pass;
* [ ] `version` is updated in `pyproject.toml` **and** `__version__`;
* [ ] `CHANGELOG.md` describes the changes;
* [ ] the README example still works;
* [ ] `git status` is clean — everything is committed;
* [ ] the release is tagged (`vX.Y.Z`) and pushed.

## **Summary**

| Concept              | Key idea                                                                 |
| -------------------- | ------------------------------------------------------------------------ |
| `src` layout         | package in `src/`, tests outside; you test what users install            |
| `pyproject.toml`     | one file for metadata, dependencies, and build configuration             |
| dependencies         | flexible ranges in the package; exact pins in `requirements.txt` for reproducibility |
| README / LICENSE     | documentation front page; legal permission to reuse                      |
| semantic versioning  | MAJOR.MINOR.PATCH communicates the impact of changes                     |
| Git tags             | mark the exact commit of each release                                    |

## **References**

Cannon, B., Ingram, D., Ganssle, P., Gedam, P., Eustace, S., Kluyver, T., & Chung, T. W. (2020). *PEP 621 — Storing project metadata in pyproject.toml*. Python Software Foundation. https://peps.python.org/pep-0621/

Chacon, S., & Straub, B. (2014). *Pro Git* (2nd ed.). Apress. https://git-scm.com/book/en/v2

Make a README. (2026). *Make a README*. https://www.makeareadme.com/

Preston-Werner, T. (2013). *Semantic versioning 2.0.0*. https://semver.org/

Python Packaging Authority. (2026). *Packaging Python projects*. https://packaging.python.org/en/latest/tutorials/packaging-projects/

Python Packaging Authority. (2026). *Python packaging user guide*. https://packaging.python.org/

Python Packaging Authority. (2026). *src layout vs flat layout*. https://packaging.python.org/en/latest/discussions/src-layout-vs-flat-layout/

Python Software Foundation. (2026). *venv — Creation of virtual environments*. https://docs.python.org/3/library/venv.html

Wilson, G., Bryan, J., Cranston, K., Kitzes, J., Nederbragt, L., & Teal, T. K. (2017). Good enough practices in scientific computing. *PLOS Computational Biology, 13*(6), e1005510. https://doi.org/10.1371/journal.pcbi.1005510